# Microsoft 365 Copilot Roadmap → Fabric Lakehouse

This notebook fetches live Copilot feature data from the **Microsoft 365 public roadmap RSS feed**, cleans and enriches it, then writes it to your Fabric Lakehouse as Delta tables.

## What gets created
| Delta Table | Description |
|---|---|
| `copilot_features` | Full feature matrix — one row per Copilot capability |
| `copilot_features_summary` | Aggregated counts by status, phase, workload, readiness |

## Prerequisites
1. Attach a **Lakehouse** to this notebook before running.
2. The notebook is self-contained — no external dependencies beyond what Fabric provides.

---
> **Data source:** `https://www.microsoft.com/releasecommunications/api/v2/m365/rss`  
> The feed is filtered to Copilot-tagged items only. Stale GA estimates (past dates on In-Development features) are automatically suppressed.

## 1 — Configuration
Edit the values in this cell before running the notebook.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────

# Name of the Delta table that will be created / overwritten in your Lakehouse
TABLE_FEATURES  = "copilot_features"
TABLE_SUMMARY   = "copilot_features_summary"

# Write mode: 'overwrite' replaces the table on every run.
# Change to 'merge' if you want incremental loads (see Section 5).
WRITE_MODE = "overwrite"

# Microsoft 365 Roadmap RSS endpoint
RSS_URL = "https://www.microsoft.com/releasecommunications/api/v2/m365/rss"
REQUEST_TIMEOUT = 30  # seconds

print("Configuration loaded.")
print(f"  Features table : {TABLE_FEATURES}")
print(f"  Summary table  : {TABLE_SUMMARY}")
print(f"  Write mode     : {WRITE_MODE}")

## 2 — Fetch the Roadmap RSS Feed

In [ ]:
import requests
import xml.etree.ElementTree as ET

ATOM_NS = "http://www.w3.org/2005/Atom"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; FabricCopilotTracker/1.0)",
    "Accept": "application/rss+xml, application/xml, text/xml, */*",
}

print(f"Fetching: {RSS_URL}")
response = requests.get(RSS_URL, headers=headers, timeout=REQUEST_TIMEOUT)
response.raise_for_status()

root    = ET.fromstring(response.content)
channel = root.find("channel")
items   = channel.findall("item")

print(f"Feed fetched successfully. Total items in feed: {len(items)}")

## 3 — Parse and Filter for Copilot Features

In [ ]:
import re
import calendar
from datetime import date, datetime

# ── Workload mapping ──────────────────────────────────────────────────────────
WORKLOAD_MAP = {
    "copilot studio":        "Copilot Studio",
    "microsoft 365 copilot": "Microsoft 365 Copilot",
    "copilot extensibility": "Copilot Extensibility",
    "microsoft 365 admin":   "Copilot Admin & Governance",
    "microsoft purview":     "Copilot Security & Compliance",
    "purview":               "Copilot Security & Compliance",
    "microsoft graph":       "Copilot Platform (Graph)",
    "microsoft teams":       "Copilot in Teams",
    "teams":                 "Copilot in Teams",
    "outlook":               "Copilot in Outlook",
    "word":                  "Copilot in Word",
    "excel":                 "Copilot in Excel",
    "powerpoint":            "Copilot in PowerPoint",
    "onenote":               "Copilot in OneNote",
    "sharepoint":            "Copilot in SharePoint",
    "onedrive":              "Copilot in OneDrive",
    "loop":                  "Copilot in Loop",
    "whiteboard":            "Copilot in Whiteboard",
    "planner":               "Copilot in Planner",
    "forms":                 "Copilot in Forms",
    "stream":                "Copilot in Stream",
    "viva insights":         "Copilot in Viva Insights",
    "viva engage":           "Copilot in Viva Engage",
    "viva learning":         "Copilot in Viva Learning",
    "viva goals":            "Copilot in Viva Goals",
    "viva connections":      "Copilot in Viva Connections",
    "viva":                  "Copilot in Viva",
    "power automate":        "Copilot in Power Automate",
    "power apps":            "Copilot in Power Apps",
    "power platform":        "Copilot in Power Platform",
    "project":               "Copilot in Project",
    "yammer":                "Copilot in Yammer",
    "microsoft search":      "Copilot in Microsoft Search",
    "microsoft entra":       "Copilot in Entra",
    "microsoft 365 apps":    "Copilot in M365 Apps",
}

PLATFORM_KEYWORDS = {"web", "desktop", "mobile", "ios", "android", "mac",
                     "windows", "teams rooms", "tablet"}

STATUS_MAP   = {"rolling out": "Rolling Out", "launched": "GA",
                "in development": "In Development"}
PHASE_ORDER  = ["frontier", "preview", "targeted release", "general availability"]
PHASE_LABEL  = {"frontier": "Frontier", "preview": "Preview",
                "targeted release": "Targeted Release",
                "general availability": "General Availability"}

MONTHS_FULL = ["january","february","march","april","may","june",
               "july","august","september","october","november","december"]
MONTHS_ABBR = ["jan","feb","mar","apr","may","jun",
               "jul","aug","sep","oct","nov","dec"]


# ── Helper functions ──────────────────────────────────────────────────────────

def clean_html(text):
    return re.sub(r"<[^>]+>", "", text or "").strip()

def is_copilot(categories):
    """Only include features Microsoft explicitly tagged as Copilot."""
    return any("copilot" in c.lower() for c in categories)

def extract_workload(title, categories):
    if ":" in title:
        product = title.split(":")[0].lower().strip()
        for key, val in WORKLOAD_MAP.items():
            if key in product:
                return val
    for cat in categories:
        for key, val in WORKLOAD_MAP.items():
            if key in cat.lower():
                return val
    return "Microsoft 365 Copilot"

def extract_status(categories):
    cats = [c.lower() for c in categories]
    for key, val in STATUS_MAP.items():
        if any(key in c for c in cats):
            return val
    return "In Development"

def extract_phase(categories):
    cats = [c.lower() for c in categories]
    for pk in PHASE_ORDER:
        if any(pk in c for c in cats):
            return PHASE_LABEL[pk]
    return "General Availability"

def extract_platforms(categories):
    result = []
    for cat in categories:
        if any(pk in cat.lower() for pk in PLATFORM_KEYWORDS):
            result.append(cat)
    return ", ".join(result)

def _date_fragment():
    MONTHS = "|".join(MONTHS_FULL + MONTHS_ABBR)
    YEAR   = r"202[4-9]|203[0-9]"
    return (
        r"(?:"
        r"(?:" + MONTHS + r")s?\.?\s+(?:" + YEAR + r")"
        r"|Q[1-4]\s*(?:CY\s*)?(?:" + YEAR + r")"
        r"|CY\s*(?:" + YEAR + r")"
        r"|H[12]\s*(?:" + YEAR + r")"
        r"|(?:first|second)\s+half\s+(?:of\s+)?(?:" + YEAR + r")"
        r"|(?:early|mid|late)\s+(?:" + YEAR + r")"
        r")"
    )

GA_CONTEXT = (
    r"(?:general\s+availability|rolling\s+out(?:\s+to\s+(?:GA|general))?|"
    r"available\s+(?:to\s+all|worldwide|generally)|"
    r"GA\s*(?:date\s*)?(?:[:\-\u2013]|\s+in\s+|\s+by\s+)|"
    r"launch(?:es|ing|ed)?(?:\s+in)?)\s*(?:[:\-\u2013])?\s*"
)

def _parse_date_raw(raw):
    raw_lower = raw.lower()
    YEAR_PAT = r"(202[4-9]|203[0-9])"
    m = re.search(r"\b(" + "|".join(MONTHS_FULL + MONTHS_ABBR) + r")\.?\s+" + YEAR_PAT, raw_lower)
    if m:
        name = m.group(1)
        if len(name) == 3 and name in MONTHS_ABBR:
            name = MONTHS_FULL[MONTHS_ABBR.index(name)]
        return f"{name.capitalize()} {m.group(2)}"
    m = re.search(r"\bQ([1-4])\s*(?:CY\s*)?" + YEAR_PAT, raw, re.IGNORECASE)
    if m: return f"Q{m.group(1)} {m.group(2)}"
    m = re.search(r"\bCY\s?" + YEAR_PAT, raw, re.IGNORECASE)
    if m: return f"CY {m.group(1)}"
    m = re.search(r"\bH([12])\s*" + YEAR_PAT, raw, re.IGNORECASE)
    if m: return f"H{m.group(1)} {m.group(2)}"
    m = re.search(r"\b(first|second)\s+half\s+(?:of\s+)?" + YEAR_PAT, raw_lower)
    if m: return f"{'H1' if m.group(1)=='first' else 'H2'} {m.group(2)}"
    m = re.search(r"\b(early|mid|late)\s+" + YEAR_PAT, raw_lower)
    if m: return f"{m.group(1).capitalize()} {m.group(2)}"
    return None

def _search_date(pattern, text):
    m = re.search(pattern, text, re.IGNORECASE)
    return _parse_date_raw(m.group(0)) if m else None

def _approx_end_date(estimate):
    """Return the END date of the stated period (used for stale-date checks)."""
    if not estimate: return None
    ym = re.search(r"(202\d|203\d)", estimate)
    if not ym: return None
    year = int(ym.group(1))
    est  = estimate.lower()
    qm = re.search(r"Q([1-4])", estimate, re.IGNORECASE)
    if qm:
        em = int(qm.group(1)) * 3
        return date(year, em, calendar.monthrange(year, em)[1])
    hm = re.search(r"H([12])", estimate, re.IGNORECASE)
    if hm:
        em = 6 if hm.group(1) == "1" else 12
        return date(year, em, calendar.monthrange(year, em)[1])
    for i, name in enumerate(MONTHS_FULL, start=1):
        if name in est:
            return date(year, i, calendar.monthrange(year, i)[1])
    if "early" in est: return date(year, 4, 30)
    if "mid"   in est: return date(year, 8, 31)
    return date(year, 12, 31)

def extract_ga_date(description, release_status):
    """Two-pass extraction: GA-context first, any date second. Suppress stale dates."""
    result = (_search_date(GA_CONTEXT + _date_fragment(), description) or
              _search_date(_date_fragment(), description))
    if result and release_status == "In Development":
        end = _approx_end_date(result)
        if end and end < date.today():
            return None  # stale — don't mislead
    return result

def derive_confidence(status, phase):
    if status in ("GA", "Rolling Out"): return "High"
    if phase == "Frontier":             return "Low"
    if phase in ("Preview", "Targeted Release"): return "Medium"
    if status == "In Development" and phase == "General Availability": return "Medium"
    return "Low"

def derive_readiness(status, phase):
    if status == "GA":           return "Safe to Promote"
    if status == "Rolling Out":
        return "Safe to Promote" if phase in ("General Availability", "Targeted Release") else "Pilot Only"
    if phase == "Frontier":      return "Do Not Commit"
    return "Pilot Only"

def parse_rss_date(s):
    for fmt in ("%a, %d %b %Y %H:%M:%S %z", "%a, %d %b %Y %H:%M:%S %Z"):
        try: return datetime.strptime(s, fmt).strftime("%Y-%m-%d %H:%M:%S")
        except: pass
    return None

def parse_iso_date(s):
    try: return datetime.fromisoformat(s.replace("Z", "+00:00")).strftime("%Y-%m-%d %H:%M:%S")
    except: return None

print("Helper functions defined.")

In [ ]:
# ── Parse all RSS items and filter for Copilot features ──────────────────────

records = []
skipped = 0

for item in items:
    guid        = (item.findtext("guid") or "").strip()
    title       = (item.findtext("title") or "").strip()
    link        = (item.findtext("link") or "").strip()
    description = clean_html(item.findtext("description") or "")
    pub_date    = parse_rss_date((item.findtext("pubDate") or "").strip())
    updated_el  = item.find(f"{{{ATOM_NS}}}updated")
    updated     = parse_iso_date((updated_el.text or "").strip()) if updated_el is not None else None
    categories  = [c.text.strip() for c in item.findall("category") if c.text]

    # Filter: only Copilot-tagged features
    if not is_copilot(categories):
        skipped += 1
        continue

    status  = extract_status(categories)
    phase   = extract_phase(categories)

    records.append({
        "feature_id":          f"ms-{guid}" if guid else None,
        "name":                title,
        "workload":            extract_workload(title, categories),
        "description":         description,
        "release_status":      status,
        "release_phase":       phase,
        "ga_estimate":         extract_ga_date(description, status),
        "confidence_level":    derive_confidence(status, phase),
        "business_readiness":  derive_readiness(status, phase),
        "platforms":           extract_platforms(categories),
        "roadmap_link":        link,
        "published_date":      pub_date,
        "last_updated":        updated,
        "source":              "MS Roadmap",
        "raw_categories":      ", ".join(categories),
        "ingested_at":         datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    })

print(f"Total feed items : {len(items)}")
print(f"Skipped (non-Copilot) : {skipped}")
print(f"Copilot features parsed : {len(records)}")

# Preview
for r in records[:3]:
    print(f"  [{r['release_status']:14s}] [{r['release_phase']:22s}] {r['workload']:30s} | {r['name'][:60]}")

## 4 — Create Spark DataFrame with Explicit Schema

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col, to_timestamp, current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("feature_id",         StringType(), True),
    StructField("name",               StringType(), True),
    StructField("workload",           StringType(), True),
    StructField("description",        StringType(), True),
    StructField("release_status",     StringType(), True),
    StructField("release_phase",      StringType(), True),
    StructField("ga_estimate",        StringType(), True),
    StructField("confidence_level",   StringType(), True),
    StructField("business_readiness", StringType(), True),
    StructField("platforms",          StringType(), True),
    StructField("roadmap_link",       StringType(), True),
    StructField("published_date",     StringType(), True),
    StructField("last_updated",       StringType(), True),
    StructField("source",             StringType(), True),
    StructField("raw_categories",     StringType(), True),
    StructField("ingested_at",        StringType(), True),
])

df = spark.createDataFrame(records, schema=schema) \
          .withColumn("published_date", to_timestamp("published_date",  "yyyy-MM-dd HH:mm:ss")) \
          .withColumn("last_updated",   to_timestamp("last_updated",    "yyyy-MM-dd HH:mm:ss")) \
          .withColumn("ingested_at",    to_timestamp("ingested_at",     "yyyy-MM-dd HH:mm:ss"))

print(f"DataFrame created: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()

In [ ]:
# Preview the data
display(
    df.select(
        "name", "workload", "release_status", "release_phase",
        "ga_estimate", "confidence_level", "business_readiness"
    ).orderBy("workload", "name")
)

## 5 — Write to Lakehouse Delta Tables

In [ ]:
# ── Write: copilot_features ───────────────────────────────────────────────────
print(f"Writing {df.count()} rows to '{TABLE_FEATURES}' (mode={WRITE_MODE})...")

df.write \
  .format("delta") \
  .mode(WRITE_MODE) \
  .option("overwriteSchema", "true") \
  .saveAsTable(TABLE_FEATURES)

print(f"✓ Table '{TABLE_FEATURES}' written successfully.")

# Verify
count = spark.table(TABLE_FEATURES).count()
print(f"  Rows in table: {count}")

In [ ]:
# ── Write: copilot_features_summary ──────────────────────────────────────────
from pyspark.sql.functions import count as cnt, round as rnd

total = df.count()

summary_df = df.groupBy("workload", "release_status", "release_phase", "business_readiness", "confidence_level") \
               .agg(cnt("*").alias("feature_count")) \
               .withColumn("pct_of_total", rnd(col("feature_count") / lit(total) * 100, 1)) \
               .withColumn("ingested_at", current_timestamp()) \
               .orderBy("workload", "release_status")

summary_df.write \
           .format("delta") \
           .mode(WRITE_MODE) \
           .option("overwriteSchema", "true") \
           .saveAsTable(TABLE_SUMMARY)

print(f"✓ Table '{TABLE_SUMMARY}' written successfully.")
display(summary_df.orderBy(col("feature_count").desc()).limit(20))

## 6 — Validate & Explore the Data

In [ ]:
features = spark.table(TABLE_FEATURES)

print("=" * 60)
print(f"Total features loaded : {features.count()}")
print("=" * 60)

print("\n── Release Status breakdown ──")
features.groupBy("release_status").count().orderBy("count", ascending=False).show()

print("── Release Phase breakdown ──")
features.groupBy("release_phase").count().orderBy("count", ascending=False).show()

print("── Business Readiness breakdown ──")
features.groupBy("business_readiness").count().orderBy("count", ascending=False).show()

print("── Top 10 Workloads ──")
features.groupBy("workload").count().orderBy("count", ascending=False).show(10)

In [ ]:
# Upcoming GA timeline — features not yet GA, with an estimate
print("── Upcoming GA timeline ──")
features.filter(
    (col("release_status") != "GA") & col("ga_estimate").isNotNull()
).groupBy("ga_estimate").count() \
 .orderBy("ga_estimate") \
 .show(20)

# Data quality: how many features have a GA estimate?
total   = features.count()
with_ga = features.filter(col("ga_estimate").isNotNull()).count()
print(f"Features with GA estimate: {with_ga}/{total} ({with_ga/total*100:.1f}%)")

In [ ]:
# Leadership view: Safe to Promote features
print("── Features safe to promote ──")
display(
    features.filter(col("business_readiness") == "Safe to Promote")
            .select("name", "workload", "release_status", "release_phase", "ga_estimate")
            .orderBy("workload", "name")
)

# Frontier / Do Not Commit features
print("── Frontier / Do Not Commit features ──")
display(
    features.filter(col("business_readiness") == "Do Not Commit")
            .select("name", "workload", "release_phase", "ga_estimate")
            .orderBy("workload")
)

## 7 — Alternative: Load from Exported Excel File

If you exported the Excel file from the Flask app and uploaded it to your Lakehouse **Files** section, use this cell instead of Sections 2–4.

1. In the Lakehouse explorer, upload your `Copilot_Feature_Matrix_*.xlsx` to `Files/`
2. Update `EXCEL_PATH` below
3. Run this cell — it will write the same `copilot_features` Delta table

In [ ]:
# ── Load from Excel export (optional alternative to RSS fetch) ────────────────
# Uncomment and update the path below, then run this cell.

# EXCEL_PATH = "Files/Copilot_Feature_Matrix_20260505_1200.xlsx"

# import pandas as pd
# from pyspark.sql.functions import current_timestamp
#
# # Read Excel with pandas (Fabric supports pandas natively)
# pdf = pd.read_excel(f"/lakehouse/default/{EXCEL_PATH}", sheet_name="Copilot Feature Matrix")
#
# # Normalise column names
# pdf.columns = [
#     c.lower().strip()
#      .replace(" ", "_")
#      .replace("/", "_")
#      .replace("-", "_")
#     for c in pdf.columns
# ]
#
# df_excel = spark.createDataFrame(pdf.astype(str).where(pdf.notna(), None)) \
#                 .withColumn("ingested_at", current_timestamp())
#
# df_excel.write \
#         .format("delta") \
#         .mode("overwrite") \
#         .option("overwriteSchema", "true") \
#         .saveAsTable(TABLE_FEATURES)
#
# print(f"✓ Loaded {df_excel.count()} rows from Excel into '{TABLE_FEATURES}'")

print("Excel loader is commented out. Uncomment and set EXCEL_PATH to use it.")

## 8 — Scheduling (Optional)

To keep the Lakehouse tables up to date automatically:

1. In Fabric, open the **Workspace** and click **+ New → Data pipeline**
2. Add a **Notebook activity** pointing to this notebook
3. Add a **Schedule trigger** (e.g., weekly on Monday 08:00)
4. The pipeline will re-run the full RSS fetch + overwrite on each trigger

Alternatively, change `WRITE_MODE = "append"` and add a deduplication step:

```python
# Deduplication after append — keep latest ingestion per feature_id
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

w   = Window.partitionBy("feature_id").orderBy(desc("ingested_at"))
deduped = spark.table(TABLE_FEATURES) \
               .withColumn("rn", row_number().over(w)) \
               .filter(col("rn") == 1).drop("rn")

deduped.write.format("delta").mode("overwrite").saveAsTable(TABLE_FEATURES)
```

---
## Summary

| Step | What happened |
|------|---------------|
| Fetch | Pulled live data from Microsoft 365 Roadmap RSS feed |
| Filter | Kept only features with an explicit **Copilot** category tag |
| Enrich | Derived workload labels, confidence level, business readiness, GA estimate |
| Validate | Suppressed stale past-date GA estimates on In-Development features |
| Write | Saved as Delta tables `copilot_features` and `copilot_features_summary` |

These tables are now queryable in Power BI, Fabric SQL endpoint, and any downstream Fabric items.